In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np


import plotly.figure_factory as ff
import plotly.graph_objects as go

In [ ]:
def profile2(theta):
    r = 10 / (np.cos(np.pi/4 - (theta % (np.pi/2))))
    return r

In [ ]:
def profile1(theta):
    a = 10 # offset
    b = 5 # amplitude
    c = 7 # number of lobes
    r = a + b * np.cos(c*theta)
    return r

In [ ]:
# Convert between Cartesian/Polar coordinates

def cart2pol(x, y):
    rho = np.sqrt(x**2 + y**2)
    phi = np.arctan2(y, x)
    return(rho, phi)

def pol2cart(rho, phi):
    x = rho * np.cos(phi)
    y = rho * np.sin(phi)
    return(x, y)

def magn(x, y):
    # magnitude of vector
    return np.sqrt(x**2 + y**2)

def rad2deg(rad):
    # convert radians to degrees
    return rad * 180 / np.pi

In [ ]:
#simulation points
n = 1000 

# Theta (radians)
T = np.linspace(1, np.pi*2 +1 , n)

# Theta degrees
TD = rad2deg(T)

In [ ]:
# Calculate Radius for each Theta
R = profile1(T)

## Working in cartesian


In [ ]:
X, Y = pol2cart(R, T)

Calculate the gradient (which in 1d case is just derivative)

In [ ]:
Gy = np.gradient(Y, X)

Check if any gradients got inf or nan

In [ ]:
sum(np.isnan(Gy))

In [ ]:
infs = np.argwhere(Gy == -np.inf)
infs

In [ ]:
infs = np.argwhere(Gy == np.inf)
infs

Optionally: set them to 0? 

In [ ]:
Gy[infs] = 0

In [ ]:
px.line(Gy)

In [ ]:
Gx = np.ones_like(Gy)

In [ ]:
L = magn(Gx, Gy)
L.shape


In [ ]:
px.scatter(L)

Calculate normals

In [ ]:
Nx, Ny = -Gy/L, Gx/L 

In [ ]:
xy  = np.array([X,Y])

In [ ]:
NxNy = np.array([Nx,Ny])

In [ ]:
NxNy.shape

Ensure all normal vectors point in the same direction 
(Relative to origin) 

In [ ]:
dot_products = np.sum(xy * NxNy, axis=0)
dot_products.shape

In [ ]:
scaling_factor_S = np.sign(dot_products)

# 3. Reshape S to be an N x 1 array to multiply correctly with N_initial (N x 2)
#    np.newaxis expands the dimension: [s1, s2, ...] -> [[s1], [s2], ...]
S_reshaped = scaling_factor_S[np.newaxis, :]

In [ ]:
S_reshaped.shape

In [ ]:
N_consistent = NxNy * S_reshaped 
N_consistent.shape

In [ ]:
Nx, Ny =  N_consistent

In [ ]:
Nx.shape

In [ ]:
# Iteration step size
d = -.3

Calculate the new points after iteration

In [ ]:
Ex, Ey =  X - d * Nx , Y - d * Ny

### Normals behavior

In [ ]:
px.line(x = X, y = Nx, hover_data= [X], width=800, height=800)

In [ ]:
px.line(x = Ny, y = Y, hover_data= [X], width=800, height=800)

In [ ]:
fig = px.line(x = X, y = Y,width=1000, height=1000, hover_data= [Gy] )
fig.add_trace(go.Scatter(x=Ex , y=Ey, mode='lines', name='with gradient'))
fig.update_xaxes(range=(-16,16))
fig.show()

In [ ]:
fig = ff.create_quiver(X,Y, - d * Nx, - d * Ny,
                        #x, y, u, v,
                       scale=.5, 
                       arrow_scale=.2,
                       name='quiver',
                       line_width=3, 
                       angle=np.pi/6,
                     #  width=800, height=800)
)

fig.add_trace(go.Scatter (x=X, y=Y,mode='lines', name='points'))

fig.update_layout(  width=800, height=800)
fig.update_xaxes(range=(-16,16))
# Add points to figure
fig.show()

# Full simulation 

In [ ]:
def step(X, Y, ):

    Gy = np.gradient(Y, X)

    #Check if any gradients got inf or nan
    assert sum(np.isnan(Gy)) == 0

    Gx = np.ones_like(Gy)

    # Calculate normals
    L = magn(Gx, Gy)
    Nx, Ny = -Gy/L, Gx/L 
    xy  = np.array([X,Y])
    NxNy = np.array([Nx,Ny])

    # Ensure all normal vectors point in the same 
    # direction (Relative to origin) 
    dot_products = np.sum(xy * NxNy, axis=0)

    scaling_factor_S = np.sign(dot_products)

    S_reshaped = scaling_factor_S[np.newaxis, :]

    N_consistent = NxNy * S_reshaped 
    Nx, Ny =  N_consistent
    return Nx, Ny


def run(func, d = -.3, steps = 1, n = 1000 ):
    # d - Iteration step size
    
    # Theta (radians)
    T = np.linspace(1, np.pi*2 +1 , n)

    # Theta degrees
    #TD = T * 180 / np.pi #  radians to degrees

    # Calculate Radius for each Theta
    R = func(T)

    X, Y = pol2cart(R, T)

    data = []
    for i in range(steps):
        Nx, Ny = step(X, Y)
        X, Y =  X - d * Nx , Y - d * Ny
        res = { 'i': i,
            # 'Theta': T, 
            # 'ThetaDeg':TD, 
            # 'Radius': R, 
            # 'Gradient': GP,
            'X': X,
            'Y': Y,
            'Nx': Nx,
            'Ny': Ny}
        data.append(res)
    return data


data = run(profile1, d = -.3, steps = 10, n = 1000 )

In [ ]:
df = [pd.DataFrame(d) for d in data]
df = pd.concat(df, ignore_index=True)

df[['R','T']] = df.apply(lambda r: cart2pol(r['X'], r['Y']), axis=1, result_type='expand')
df['TD']      = df.apply(lambda r: rad2deg(r['T']), axis=1)

In [ ]:
df

I need a python function that solves the following:
I have an array of size n of 2d vectors. Each vector has an origin X,Y and direction Nx, Ny.
If that helps, you may assume that all vectors origins are on some curve and point in directions orthogonal to the curve.
Order of vectors matters. 
For each pair of vectors adjacent within the array, I want to find their intersection point. 
No need to check for intersections of all possible pairs of vectors. Just the adjacent ones. 
Only intersections in the direction of vectors matter. I don't care if they intersect behind vectors. Does that benefit performance?
Use python/numpy. Prefer vector operations, try to avoid for loops.
Currently X,Y,Nx,Ny are separate variables, but it's no problem to put them together if that benefits performance

If we introduce maximum distance from origin within which the intersections matter, does it help performance? 

## Interactive Polar plot

In [ ]:
fig = px.line_polar(df, r="R", theta="TD", line_close=True,
                    range_r=[0,20], animation_frame="i",
                    #color_discrete_sequence=px.colors.sequential.Plasma_r,
                    #template="plotly_dark",)               
                    width=600, height=600
                    )
fig.show()

In [ ]:
fig = px.line(df, y="R", x="TD", #line_close=True,
                    #color_discrete_sequence=px.colors.sequential.Plasma_r,
                    #template="plotly_dark",)
                    width=1000, height=600,
                    animation_frame="i"
                    )
fig.show()

# Vector intersections

In [ ]:
def adjacent_intersections(X, Y, Nx, Ny, forward_only=True, eps=1e-12):
    """Compute intersections for adjacent 2D rays (vectorized)."""
    X = np.asarray(X).ravel()
    Y = np.asarray(Y).ravel()
    Nx = np.asarray(Nx).ravel()
    Ny = np.asarray(Ny).ravel()
    if not (X.size == Y.size == Nx.size == Ny.size):
        raise ValueError('All inputs must have same length')
    # Stack as (n,2) arrays and form adjacent pairs (n-1)
    c = np.stack((X, Y), axis=1)  # (n,2)
    v = np.stack((Nx, Ny), axis=1)
    c_i = c[:-1]; c_j = c[1:]
    v_i = v[:-1]; v_j = v[1:]
    C = c_j - c_i
    # 2D cross product (scalar) for arrays of shape (m,2)
    def cross2(a, b):
        return a[:, 0] * b[:, 1] - a[:, 1] * b[:, 0]
    den = cross2(v_i, v_j)
    num_ti = cross2(C, v_j)
    num_tj = cross2(C, v_i)
    # safe division with NumPy (will produce inf/nan where den==0)
    with np.errstate(divide='ignore', invalid='ignore'):
        ti = num_ti / den
        tj = num_tj / den
    # validity mask: non-parallel and finite
    valid = np.isfinite(den) & (np.abs(den) > eps)
    if forward_only:
        valid &= (ti > 0) & (tj > 0)
    # Intersection points computed from ray i: p = c_i + ti * v_i
    Px = c_i[:, 0] + ti * v_i[:, 0]
    Py = c_i[:, 1] + ti * v_i[:, 1]
    # Mask invalid entries as NaN for clarity
    invalid = ~valid
    if invalid.any():
        Px = Px.astype(float)
        Py = Py.astype(float)
        ti = ti.astype(float)
        tj = tj.astype(float)
        Px[invalid] = np.nan
        Py[invalid] = np.nan
        ti[invalid] = np.nan
        tj[invalid] = np.nan
    return Px, Py, ti, tj, valid

In [ ]:
v1 = np.array([-4, -2]).T
c1 = np.array([6, 3]).T

v2 = np.array([5, -2]).T
c2 = np.array([-3, 3]).T

v3 = np.array([-1, 3]).T
c3 = np.array([4, -2]).T

# in this case the solved x is [-1.  1.], error is 0, and rank is 2i

In [ ]:

wat = np.linalg.lstsq(A, b)

x, err, rank = wat[:3]
if rank == 2:
    # intersection exists
    i1 = v1 * x[0] + c1
    print(i1)
else:
    print("no intersection")

In [ ]:
cx,cy = np.array([c1, c2, c3]).T
cx,cy

In [ ]:
vx,vy = np.array([v1, v2, v3]).T
vx,vy

In [ ]:
i1

In [ ]:



fig = ff.create_quiver(cx, cy, vx, vy,
                        #x, y, u, v,
                       scale=1, 
                       arrow_scale=.2,
                       name='quiver',
                       line_width=3, 
                       angle=np.pi/6,
                     #  width=800, height=800)
)
fig.add_trace(go.Scatter (x=[i1[0]], y=[i1[1]]))

In [ ]:
# Fit a line, y = mx + c, through some noisy data-points:

x = np.array([0, 1, 2, 3, 6, 8])
y = np.array([-1, 0.2, 0.9, 2.1, 3, 7])
# By examining the coefficients, we see that the line should have a gradient of roughly 1 and cut the y-axis at, more or less, -1.

# We can rewrite the line equation as y = Ap, where A = [[x 1]] and p = [[m], [c]]. Now use lstsq to solve for p:

A = np.vstack([x, np.ones(len(x))]).T
# >>> A
# array([[ 0.,  1.],
#        [ 1.,  1.],
#        [ 2.,  1.],
#        [ 3.,  1.]])
A.shape

In [ ]:
y.shape

In [ ]:


sol, res, rank, s = np.linalg.lstsq(A, y, rcond=None)
m, c = sol
m, c

In [ ]:
sol, res, rank, s

In [ ]:
i1 = v1 * x[0] + c1

In [ ]:
A.shape

In [ ]:
wat

In [ ]:
px.scatter(x=x, y=y)    